Imports

In [1]:
import pandas as pd

Configuration

In [ ]:
df = pd.read_csv("aso_inhibitions_with_canonical_gene.csv")
col1 = "target_gene"
col2 = "Canonical Gene Name"

Check unfound canonical genes

In [ ]:
not_found = df[df[col2].isna()]

print(f"{len(not_found)} RNA contexts were not found in the genome")

not_found.to_csv("rna_contexts_not_found.csv", index=False)

Find targets with multiple possible genes

In [ ]:
multi_gene_mask = (
    df["Canonical Gene Name"]
    .notna()
    & df["Canonical Gene Name"].str.contains(";", regex=False)
)

count_multi = multi_gene_mask.sum()

print(f"{count_multi} rows have more than one canonical gene")

In [ ]:
df[multi_gene_mask]

In [ ]:

fixed_df = df.copy()

multi_gene_mask = (
        fixed_df[col2].notna()
        & fixed_df[col2].str.contains(";", regex=False)
)

print(f"Fixing {multi_gene_mask.sum()} rows with multiple canonical genes")

# Replace Canonical Gene Name with target_gene where ambiguous
fixed_df.loc[multi_gene_mask, col2] = fixed_df.loc[multi_gene_mask, col1]

# Save fixed version
fixed_df.to_csv(
    "aso_inhibitions_with_canonical_gene_fixed.csv",
    index=False
)

print("Saved aso_inhibitions_with_canonical_gene_fixed.csv")

function that checks for mismatches between targets and canonical gene names

In [ ]:
def check_baddies(df, col_1, col_2):
    baddies = df[df[col_1] != df[col_2]][col_1].unique()
    canonical = df[df[col_1] != df[col_2]][col_2].unique()
    new_df = pd.DataFrame()
    new_df["baddies"] = baddies
    new_df["canonical"] = canonical
    return new_df

execution

In [ ]:
new = check_baddies(df, col1, col2)
new.to_csv("canonical_baddies.csv", index=False)

In [ ]:
new

find cell line for baddies